# Robust View-Bagging Ensemble

This experiment combines feature-view bagging and training-window bagging to reduce dependence on any single representation or historical period. The final prediction averages percentile ranks across retained models, balancing contributions from the raw, graph and full feature views. The notebook evaluates forward-chaining ROC-AUC and generates the competition submission.

The model selection rule excludes only models whose weighted validation ROC-AUC falls more than 0.015 below the strongest individual model. No blend weights are tuned on the validation folds.


In [1]:
import warnings, time, gc
warnings.filterwarnings('ignore')

from scipy import sparse
from scipy.sparse.csgraph import connected_components
from scipy.stats import rankdata
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

# ----------------------------- CONFIG -----------------------------
SEED         = 42
FAST_MODE    = False      # True = quick smoke test (fewer trees, lower AUC)
N_JOBS       = -1
TOP_K_AGG    = 20         # features used for neighbour / component aggregates
TOP_K_REL    = 30         # features used for within-step rank / z-score
RECENT       = 15         # "recent" window = last RECENT labelled steps before the cut
CV_SEEDS     = 1
FINAL_SEEDS  = 3
FLOOR        = 0.015      # drop a model only if its CV AUC is this far below the best model
t0 = time.time()

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

ROOT = next((q for q in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (q / 'data' / 'raw' / 'train.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Cannot locate data/raw/train.csv')
DATA = ROOT/'data'/'raw'
train = pd.read_csv(DATA/'train.csv')
test = pd.read_csv(DATA/'test.csv')
sample = pd.read_csv(DATA/'sample_submission.csv')
features = [c for c in train if c.startswith('feat_')]
labelled = train.dropna(subset=['label']).copy()
assert labelled.label.isin([0,1]).all()
assert {'time_step','txId'}.issubset(train.columns)
assert all(c in test.columns for c in features)
print('Labelled:',len(labelled),'Test:',len(test),'Features:',len(features))
print('Labelled time:',labelled.time_step.min(),labelled.time_step.max())

Labelled: 31235 Test: 15329 Features: 165
Labelled time: 1 35


## Data and transaction graph

Combine training and test rows for label-free graph and time-step feature construction. Test labels are not accessed.


In [3]:
test_ = test.copy()
test_['label'] = np.nan                       # never read test labels
if 'time_step' not in test_.columns: test_['time_step'] = np.nan
if 'txId' not in test_.columns:      test_['txId'] = np.nan

cols = ['txId', 'time_step', 'label'] + features
ALL = pd.concat([train[cols].assign(is_test=0), test_[cols].assign(is_test=1)], ignore_index=True)
N = len(ALL)
TEST_POS = np.where(ALL.is_test.values == 1)[0]
assert len(TEST_POS) == len(test)

# ---- edge list ----
edge_files = sorted(DATA.glob('*edge*.csv'))
print('Edge files found:', [p.name for p in edge_files])
src = dst = np.array([], dtype=np.int64)
USE_GRAPH = False
if edge_files and ALL.loc[ALL.is_test == 1, 'txId'].notna().all():
    assert ALL.txId.is_unique, 'txId must be unique across train+test'
    edges = pd.read_csv(edge_files[0])
    id2pos = pd.Series(np.arange(N), index=ALL.txId.values)
    s = edges.iloc[:, 0].map(id2pos); d = edges.iloc[:, 1].map(id2pos)
    ok = s.notna() & d.notna() & (s != d)
    print(f'Edges: {len(edges):,} | mapped to known nodes: {ok.mean():.1%}')
    src, dst = s[ok].astype(np.int64).values, d[ok].astype(np.int64).values
    USE_GRAPH = len(src) > 0
print('USE_GRAPH =', USE_GRAPH)

# ---- time steps for rows that lack one (e.g. test) ----
# Connected transactions share a time step, so propagate it through connected components.
if USE_GRAPH:
    Ab_tmp = sparse.csr_matrix((np.ones(len(src), dtype=np.float32), (src, dst)), shape=(N, N))
    _, comp_tmp = connected_components(Ab_tmp, directed=False)
    comp_step = ALL.groupby(comp_tmp)['time_step'].transform('min')   # min of known steps
    ALL['step'] = ALL['time_step'].fillna(comp_step)
    del Ab_tmp
else:
    ALL['step'] = ALL['time_step']
print('Rows with unknown step (0 = unknown group):', int(ALL.step.isna().sum()))
ALL['step'] = ALL['step'].fillna(0).astype(int)

lab_mask = (ALL.is_test.values == 0) & ALL.label.notna().values
LAB_ROWS = np.where(lab_mask)[0]
Y_ALL = ALL.label.values
STEP = ALL.step.values
M = int(STEP[LAB_ROWS].max())
print('Last labelled step M =', M, '| test steps:', sorted(set(STEP[TEST_POS]))[:3], '...')

Edge files found: ['txs_edgelist.csv']
Edges: 234,355 | mapped to known nodes: 75.1%
USE_GRAPH = True
Rows with unknown step (0 = unknown group): 0
Last labelled step M = 35 | test steps: [np.int64(36), np.int64(37), np.int64(38)] ...


## Forward-chaining folds and feature selection

Train on earlier labelled time steps and evaluate on later labelled steps. Feature importance is estimated from an early training window.


In [4]:
smin = int(STEP[LAB_ROWS].min())
# Far-horizon folds: train <= c, validate on ALL labelled steps after c (pooled), like the real test.
CUTS = [c for c in (M - 15, M - 10, M - 5) if c >= smin + 4]
FOLD_W = np.array([M - c for c in CUTS], dtype=float)          # weight folds by horizon
print('Forward CV cuts:', CUTS, '| validation steps: c+1..%d | fold weights: %s' % (M, np.round(FOLD_W / FOLD_W.sum(), 2)))

sel_rows = LAB_ROWS[STEP[LAB_ROWS] <= CUTS[0]]
imp = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, colsample_bytree=0.5,
                     subsample=0.8, subsample_freq=1, random_state=SEED, n_jobs=N_JOBS,
                     verbose=-1, importance_type='gain')
imp.fit(ALL.loc[sel_rows, features], Y_ALL[sel_rows].astype(int))
rank_feats = pd.Series(imp.feature_importances_, index=features).sort_values(ascending=False).index.tolist()
agg_feats, rel_feats = rank_feats[:TOP_K_AGG], rank_feats[:TOP_K_REL]
print('Top features:', rank_feats[:10])

Forward CV cuts: [20, 25, 30] | validation steps: c+1..35 | fold weights: [0.5  0.33 0.17]
Top features: ['feat_90', 'feat_53', 'feat_47', 'feat_49', 'feat_55', 'feat_41', 'feat_142', 'feat_5', 'feat_2', 'feat_60']


## Graph features

Construct structural and neighbouring-transaction features without using class labels.


In [5]:
def longest_path(s, d, n, max_iter=300):
    """Longest path ending at each node (edges s -> d) by iterative relaxation."""
    depth = np.zeros(n, dtype=np.int32)
    for _ in range(max_iter):
        nd = depth.copy()
        np.maximum.at(nd, d, depth[s] + 1)
        if np.array_equal(nd, depth): break
        depth = nd
    return depth

def pagerank(Ab, alpha=0.85, iters=40):
    n = Ab.shape[0]
    out = np.asarray(Ab.sum(1)).ravel()
    P = sparse.diags(np.where(out > 0, 1.0 / np.maximum(out, 1), 0.0)) @ Ab
    r = np.full(n, 1.0 / n)
    dangling = out == 0
    for _ in range(iters):
        r = alpha * (P.T @ r + r[dangling].sum() / n) + (1 - alpha) / n
    return r * n

def build_graph_features(ALL, src, dst, feats):
    n = len(ALL)
    A = sparse.csr_matrix((np.ones(len(src), dtype=np.float32), (src, dst)), shape=(n, n))
    w_out = np.asarray(A.sum(1)).ravel(); w_in = np.asarray(A.sum(0)).ravel()   # with multiplicity
    Ab = A.copy(); Ab.data[:] = 1.0
    out_nb = np.asarray(Ab.sum(1)).ravel(); in_nb = np.asarray(Ab.sum(0)).ravel()
    G = {}
    G['g_in_deg'], G['g_out_deg'] = w_in, w_out
    G['g_in_nb'], G['g_out_nb'] = in_nb, out_nb
    G['g_deg_ratio'] = (in_nb + 1) / (out_nb + 1)
    G['g_paths2_out'] = Ab @ out_nb
    G['g_paths2_in'] = Ab.T @ in_nb
    G['g_depth'] = longest_path(src, dst, n)
    G['g_height'] = longest_path(dst, src, n)
    G['g_pr_fwd'] = pagerank(Ab)
    G['g_pr_rev'] = pagerank(Ab.T.tocsr())
    n_comp, comp = connected_components(Ab, directed=False)
    G['g_comp_size'] = np.bincount(comp)[comp]
    G = pd.DataFrame(G, index=ALL.index)

    X = ALL[feats].to_numpy(dtype=np.float32)
    Xf = np.where(np.isnan(X), np.nanmedian(X, axis=0), X).astype(np.float32)
    Pin = sparse.diags(1.0 / np.maximum(in_nb, 1)) @ Ab.T.tocsr()
    Pout = sparse.diags(1.0 / np.maximum(out_nb, 1)) @ Ab
    blocks = [G]

    def mk(prefix, arr, valid):
        arr = np.asarray(arr, dtype=np.float32).copy()
        arr[~valid] = np.nan
        return pd.DataFrame(arr, columns=[f'{prefix}_{f}' for f in feats], index=ALL.index)

    m_in, m_out = Pin @ Xf, Pout @ Xf
    blocks += [mk('g_inmean', m_in, in_nb > 0), mk('g_outmean', m_out, out_nb > 0),
               mk('g_inmean2', Pin @ m_in, G['g_paths2_in'].values > 0),
               mk('g_outmean2', Pout @ m_out, G['g_paths2_out'].values > 0)]

    def edge_agg(keys, vals, prefix):
        df = pd.DataFrame(vals, columns=feats); df['_k'] = keys
        g = df.groupby('_k')
        res = []
        for fn in ('max', 'min'):
            a = getattr(g, fn)().reindex(np.arange(n)).astype(np.float32)
            a.columns = [f'{prefix}_{fn}_{f}' for f in feats]; a.index = ALL.index
            res.append(a)
        return res
    blocks += edge_agg(dst, X[src], 'g_in')
    blocks += edge_agg(src, X[dst], 'g_out')

    cm = pd.DataFrame(Xf).groupby(comp).transform('mean').to_numpy(dtype=np.float32)
    blocks += [mk('g_cmean', cm, np.ones(n, bool)), mk('g_cdiff', Xf - cm, np.ones(n, bool))]
    out = pd.concat(blocks, axis=1).astype('float32')
    return out, Ab

t1 = time.time()
if USE_GRAPH:
    G, Ab = build_graph_features(ALL, src, dst, agg_feats)
else:
    G, Ab = pd.DataFrame(index=ALL.index), sparse.csr_matrix((N, N), dtype=np.float32)
print(f'Graph features: {G.shape[1]} columns in {time.time()-t1:.0f}s')

Graph features: 212 columns in 1s


## Within-step relative features

Calculate percentile ranks and standardised values within each time step.


In [6]:
def step_relative_features(ALL, feats):
    g = ALL.groupby('step')[feats]
    rk = g.rank(pct=True).astype('float32'); rk.columns = [f'{c}__rk' for c in feats]
    med = g.transform('median'); sd = g.transform('std').replace(0, np.nan)
    z = ((ALL[feats] - med) / sd).astype('float32'); z.columns = [f'{c}__z' for c in feats]
    return pd.concat([rk, z], axis=1)

R = step_relative_features(ALL, rel_feats)
FULL = pd.concat([ALL[features].astype('float32'), G, R], axis=1)
assert FULL.columns.is_unique
FS = {'raw': features,
      'graph': features + list(G.columns),
      'full': list(FULL.columns)}
print({k: len(v) for k, v in FS.items()}, '| FULL matrix:', FULL.shape)
del R; gc.collect()

{'raw': 165, 'graph': 377, 'full': 437} | FULL matrix: (157101, 437)


20

## Feature views and base models


In [7]:
HAS_GRAPH = G.shape[1] > 0
VIEW_COLS = {'raw': list(features), 'full': FS['full']}
if HAS_GRAPH: VIEW_COLS['graph'] = FS['graph']
print({k: len(v) for k, v in VIEW_COLS.items()})

def make_lgbm(seed):
    return LGBMClassifier(n_estimators=250 if FAST_MODE else 800, learning_rate=0.03, num_leaves=24,
                          min_child_samples=30, subsample=0.8, subsample_freq=1, colsample_bytree=0.35,
                          reg_lambda=3.0, random_state=seed, n_jobs=N_JOBS, verbose=-1)
def make_cat(seed):
    return CatBoostClassifier(iterations=250 if FAST_MODE else 700, learning_rate=0.05, depth=6,
                              l2_leaf_reg=5, rsm=0.3, eval_metric='AUC', random_seed=seed,
                              verbose=0, thread_count=N_JOBS, allow_writing_files=False)
def make_rf(seed):
    return RandomForestClassifier(n_estimators=150 if FAST_MODE else 400, max_features='sqrt',
                                  min_samples_leaf=2, n_jobs=N_JOBS, random_state=seed)
def make_et(seed):
    return ExtraTreesClassifier(n_estimators=150 if FAST_MODE else 400, max_features='sqrt',
                                min_samples_leaf=1, n_jobs=N_JOBS, random_state=seed)
ALGO = {'lgbm': make_lgbm, 'cat': make_cat, 'rf': make_rf, 'et': make_et}

SPECS = {   # name: (algorithm, view, training window)
    'raw_lgbm':        ('lgbm', 'raw',   'all'),
    'raw_cat':         ('cat',  'raw',   'all'),
    'raw_rf':          ('rf',   'raw',   'all'),
    'raw_et':          ('et',   'raw',   'all'),
    'raw_lgbm_recent': ('lgbm', 'raw',   'recent'),
    'gr_lgbm':         ('lgbm', 'graph', 'all'),
    'gr_rf':           ('rf',   'graph', 'all'),
    'gr_lgbm_recent':  ('lgbm', 'graph', 'recent'),
    'full_lgbm':       ('lgbm', 'full',  'all'),
    'full_rf':         ('rf',   'full',  'all'),
}
SPECS = {k: v for k, v in SPECS.items() if v[1] in VIEW_COLS}
VIEW_OF = {k: v[1] for k, v in SPECS.items()}

def fit_spec(name, tr_rows, y, pool_rows, ref, n_seeds=1):
    algo, view, window = SPECS[name]
    if window == 'recent':
        keep = STEP[tr_rows] > ref - RECENT
        tr_rows, y = tr_rows[keep], y[keep]
    cols = VIEW_COLS[view]
    Xtr, Xp = FULL.iloc[tr_rows][cols], FULL.iloc[pool_rows][cols]
    if algo in ('rf', 'et'): Xtr, Xp = Xtr.fillna(-999), Xp.fillna(-999)
    n_seeds = 1 if algo in ('rf', 'et') else n_seeds
    out = np.zeros(len(pool_rows))
    for s in range(n_seeds):
        m = ALGO[algo](SEED + 17 * s); m.fit(Xtr, y)
        out += m.predict_proba(Xp)[:, 1]
    return out / n_seeds

def rank01(x): return rankdata(x) / len(x)

def balanced_blend(preds, names):
    """Rank-average with each VIEW getting equal total weight."""
    views = sorted({VIEW_OF[n] for n in names})
    out = 0
    for v in views:
        ns = [n for n in names if VIEW_OF[n] == v]
        out = out + np.mean([rank01(preds[n]) for n in ns], axis=0)
    return out / len(views)

def view_blend(preds, names, view):
    ns = [n for n in names if VIEW_OF[n] == view]
    return np.mean([rank01(preds[n]) for n in ns], axis=0)

def fold_rows(c):
    """Train: labelled steps <= c. Pool: every non-test row (labelled + unlabelled) in steps c+1..M."""
    tr = LAB_ROWS[STEP[LAB_ROWS] <= c]
    pool = np.where((ALL.is_test.values == 0) & (STEP > c) & (STEP <= M))[0]
    labm = ~np.isnan(Y_ALL[pool])
    return tr, Y_ALL[tr].astype(int), pool, labm, Y_ALL[pool][labm].astype(int)

def wmean(a): return float(np.average(a, weights=FOLD_W))
unl_after = np.where((ALL.is_test.values == 0) & np.isnan(Y_ALL) & (STEP > M))[0]
POOL = np.r_[TEST_POS, unl_after]
n_test = len(TEST_POS)
print(f'models: {len(SPECS)} | final pool: {len(POOL):,} (test {n_test:,} + unlabelled {len(unl_after):,})')

{'raw': 165, 'full': 437, 'graph': 377}
models: 10 | final pool: 15,329 (test 15,329 + unlabelled 0)


## Distribution-shift diagnostic

Measure how easily a classifier distinguishes historical and future observations within each feature view.


In [8]:
def adversarial_audit(cols, n_max=10000):
    rng = np.random.RandomState(SEED)
    past = np.where((ALL.is_test.values == 0) & (STEP > 0) & (STEP <= M))[0]
    a = rng.choice(past, min(n_max, len(past)), replace=False)
    b = rng.choice(POOL, min(n_max, len(POOL)), replace=False)
    X = pd.concat([FULL.iloc[a][cols], FULL.iloc[b][cols]]).reset_index(drop=True)
    yv = np.r_[np.zeros(len(a)), np.ones(len(b))]
    idx = rng.permutation(len(X)); h = len(X) // 2
    m = LGBMClassifier(n_estimators=150, learning_rate=0.1, num_leaves=15, colsample_bytree=0.5,
                       subsample=0.8, subsample_freq=1, random_state=SEED, n_jobs=N_JOBS,
                       verbose=-1, importance_type='gain')
    m.fit(X.iloc[idx[:h]], yv[idx[:h]])
    auc = roc_auc_score(yv[idx[h:]], m.predict_proba(X.iloc[idx[h:]])[:, 1])
    top = pd.Series(m.feature_importances_, index=cols).sort_values(ascending=False).head(8)
    return auc, top

for v, cols_ in VIEW_COLS.items():
    auc, top = adversarial_audit(cols_)
    print(f'[{v:5s}] adversarial AUC past-vs-future = {auc:.3f} | most drifting: {list(top.index[:6])}')

[raw  ] adversarial AUC past-vs-future = 1.000 | most drifting: ['feat_101', 'feat_136', 'feat_100', 'feat_103', 'feat_139', 'feat_2']
[full ] adversarial AUC past-vs-future = 1.000 | most drifting: ['g_cmean_feat_100', 'feat_101', 'g_comp_size', 'g_cmean_feat_103', 'feat_100', 'feat_136']
[graph] adversarial AUC past-vs-future = 1.000 | most drifting: ['g_cmean_feat_100', 'g_comp_size', 'g_cmean_feat_103', 'feat_136', 'g_out_max_feat_103', 'g_in_max_feat_103']


## Forward-chaining evaluation

Report pooled ROC-AUC over later labelled time steps. A fixed performance floor filters substantially weaker base models; ensemble weights remain equal across views.


In [9]:
cv = []
for c in CUTS:
    print(f'--- fold: train <= {c}, validate {c+1}..{M}')
    tr, y, pool, labm, yv = fold_rows(c)
    f = dict(c=c, pool=pool, labm=labm, yv=yv, p={})
    for n in SPECS:
        ts = time.time()
        f['p'][n] = fit_spec(n, tr, y, pool, ref=c, n_seeds=CV_SEEDS)
        print(f'   {n:16s} {time.time()-ts:5.0f}s')
    cv.append(f)

def fold_auc(f, score): return roc_auc_score(f['yv'], score[f['labm']])
tab = pd.DataFrame([dict(model=n, view=VIEW_OF[n], **{f'fold{f["c"]}': fold_auc(f, f['p'][n]) for f in cv})
                    for n in SPECS])
tab['wmean'] = [wmean([fold_auc(f, f['p'][n]) for f in cv]) for n in SPECS]
tab = tab.sort_values('wmean', ascending=False).round(4)
display(tab)

KEPT = [n for n in SPECS if tab.set_index('model').loc[n, 'wmean'] >= tab.wmean.max() - FLOOR]
dropped = [n for n in SPECS if n not in KEPT]
print('Dropped by sanity floor:', dropped if dropped else 'none')

def report(label, fn):
    a = [fold_auc(f, fn(f)) for f in cv]
    print(f'{label:34s} folds {np.round(a, 4)} | wmean {wmean(a):.4f}')
    return wmean(a)

print()
for v in VIEW_COLS:
    ns = [n for n in KEPT if VIEW_OF[n] == v]
    if ns: report(f'view = {v}', lambda f, ns=ns, v=v: view_blend(f['p'], ns, v))
report('equal-weight, all models', lambda f: np.mean([rank01(f['p'][n]) for n in SPECS], axis=0))
CV_ROBUST = report('VIEW-BALANCED (recommended)', lambda f: balanced_blend(f['p'], KEPT))

--- fold: train <= 20, validate 21..35
   raw_lgbm             4s
   raw_cat              5s
   raw_rf               3s
   raw_et               1s
   raw_lgbm_recent      4s
   gr_lgbm              7s
   gr_rf                4s
   gr_lgbm_recent       6s
   full_lgbm            8s
   full_rf              7s
--- fold: train <= 25, validate 26..35
   raw_lgbm             8s
   raw_cat             11s
   raw_rf               5s
   raw_et               2s
   raw_lgbm_recent     13s
   gr_lgbm             18s
   gr_rf                7s
   gr_lgbm_recent      12s
   full_lgbm           24s
   full_rf             12s
--- fold: train <= 30, validate 31..35
   raw_lgbm            15s
   raw_cat              9s
   raw_rf               6s
   raw_et               3s
   raw_lgbm_recent      9s
   gr_lgbm             27s
   gr_rf               12s
   gr_lgbm_recent      21s
   full_lgbm           16s
   full_rf             13s


,model,view,fold20,fold25,fold30,wmean
4,raw_lgbm_recent,raw,0.8778,0.9975,0.9981,0.9377
0,raw_lgbm,raw,0.8708,0.9967,0.9984,0.9341
8,full_lgbm,full,0.8598,0.9930,0.9964,0.9270
5,gr_lgbm,graph,0.8477,0.9959,0.9973,0.9220
7,gr_lgbm_recent,graph,0.8444,0.9960,0.9978,0.9205
9,full_rf,full,0.8273,0.9903,0.9944,0.9095
1,raw_cat,raw,0.8201,0.9966,0.9987,0.9087
3,raw_et,raw,0.8215,0.9938,0.9955,0.9079
2,raw_rf,raw,0.8061,0.9933,0.9966,0.9002
6,gr_rf,graph,0.8074,0.9912,0.9952,0.9000


Dropped by sanity floor: ['raw_cat', 'raw_rf', 'raw_et', 'gr_lgbm', 'gr_rf', 'gr_lgbm_recent', 'full_rf']

view = raw                         folds [0.8746 0.9975 0.9984] | wmean 0.9362
view = full                        folds [0.8598 0.993  0.9964] | wmean 0.9270
equal-weight, all models           folds [0.8335 0.9971 0.9984] | wmean 0.9155
VIEW-BALANCED (recommended)        folds [0.8661 0.9965 0.9982] | wmean 0.9316


## Per-time-step diagnostics

Compare pooled and within-step ROC-AUC to examine differences in performance across periods.


In [10]:
f = cv[0]
sc = balanced_blend(f['p'], KEPT)
d = pd.DataFrame({'step': STEP[f['pool']], 'y': Y_ALL[f['pool']], 's': sc}).dropna(subset=['y'])
rows_ = []
for st, g in d.groupby('step'):
    rows_.append(dict(step=int(st), n=len(g), illicit_rate=round(g.y.mean(), 3),
                      auc=round(roc_auc_score(g.y, g.s), 4) if 0 < g.y.sum() < len(g) else np.nan,
                      mean_score=round(g.s.mean(), 3)))
diag = pd.DataFrame(rows_)
display(diag)
print(f"fold c={f['c']}: pooled AUC {roc_auc_score(d.y, d.s):.4f} | mean per-step AUC {diag.auc.mean():.4f}")

,step,n,illicit_rate,auc,mean_score
0,21,641,0.156,0.9941,0.382
1,22,1763,0.090,0.9991,0.247
2,23,1187,0.045,0.9496,0.201
3,24,1126,0.122,0.7242,0.269
4,25,594,0.199,0.9997,0.350
5,26,517,0.186,0.9199,0.408
6,27,206,0.117,0.9984,0.343
7,28,284,0.299,1.0000,0.441
8,29,1174,0.280,0.6698,0.309
9,30,524,0.158,0.9940,0.346


fold c=20: pooled AUC 0.8661 | mean per-step AUC 0.9096


## Final training and test predictions

Fit retained models on all available labelled training rows and generate test predictions.


In [11]:
TR, YTR = LAB_ROWS, Y_ALL[LAB_ROWS].astype(int)
print(f'Train rows: {len(TR):,}')
final_p = {}
for n in KEPT:
    ts = time.time()
    final_p[n] = fit_spec(n, TR, YTR, POOL, ref=M, n_seeds=FINAL_SEEDS)
    print(f'   {n:16s} {time.time()-ts:5.0f}s')

robust_score = balanced_blend(final_p, KEPT)[:n_test]
view_scores = {v: view_blend(final_p, KEPT, v)[:n_test] for v in VIEW_COLS if any(VIEW_OF[n] == v for n in KEPT)}
for v, s_ in view_scores.items():
    print(f'Spearman(robust, view={v}) =', round(pd.Series(robust_score).corr(pd.Series(s_), method='spearman'), 4))

Train rows: 31,235
   raw_lgbm            56s
   raw_lgbm_recent     26s
   full_lgbm           78s
Spearman(robust, view=raw) = 0.9569
Spearman(robust, view=full) = 0.9611


## Submission files and diagnostics


In [12]:
def write_submission(score, path):
    sub = sample.copy()
    id_col, tgt_col = sub.columns[0], sub.columns[1]
    if id_col in test.columns:
        sub[tgt_col] = sub[id_col].map(dict(zip(test[id_col], score)))
    else:
        assert len(sub) == len(test), 'sample_submission and test differ in length'
        sub[tgt_col] = np.asarray(score)
    assert sub[tgt_col].notna().all() and sub[tgt_col].between(0, 1).all()
    path.parent.mkdir(parents=True, exist_ok=True)
    sub.to_csv(path, index=False)
    print('saved', path, sub.shape)
    return sub

SUB_DIR = ROOT / 'submissions'
sub_robust = write_submission(robust_score, SUB_DIR / 'sub_14_robust.csv')
for v, s_ in view_scores.items():
    write_submission(s_, SUB_DIR / f'sub_14_view_{v}.csv')

(ROOT / 'outputs').mkdir(exist_ok=True)
tab.to_csv(ROOT / 'outputs' / '14_cv_table.csv', index=False)
diag.to_csv(ROOT / 'outputs' / '14_step_diagnostic.csv', index=False)
print(f'\nKept models: {KEPT}')
print(f'Far-horizon CV AUC (weighted) of the view-balanced ensemble: {CV_ROBUST:.4f}')
print(f'Total runtime: {(time.time()-t0)/60:.1f} min')
sub_robust.head()

saved /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4012/bt4012-fraud-detection/submissions/sub_14_robust.csv (15329, 2)
saved /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4012/bt4012-fraud-detection/submissions/sub_14_view_raw.csv (15329, 2)
saved /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4012/bt4012-fraud-detection/submissions/sub_14_view_full.csv (15329, 2)

Kept models: ['raw_lgbm', 'raw_lgbm_recent', 'full_lgbm']
Far-horizon CV AUC (weighted) of the view-balanced ensemble: 0.9316
Total runtime: 7.7 min


,index,target
0,0,0.418423
1,1,0.715034
2,2,0.632184
3,3,0.500718
4,4,0.534689
